# 04 · Predicciones semanales y registro

**Ejecutar cada semana antes de los partidos** (por ejemplo, el martes o miércoles, y de nuevo el domingo en la mañana si hay noticias de lesiones).

1. Actualiza los datos de nflverse y arma las filas de la semana siguiente con los rosters activos y el calendario.
2. Entrena con **todos** los partidos ya jugados, con los hiperparámetros congelados de `config/model.yaml`, y predice la media y el **rango P10–P90**: regresión por cuantiles con la calibración de `config/ranges.yaml` (notebook 05).
3. Descarga la proyección de ESPN de la semana (jugadores con roster y agentes libres) y el estado de lesión.
4. Agrega todo a **`data/predictions_log/<temporada>.csv`**. Solo se agregan filas y solo se registran predicciones de partidos que aún no empiezan. ESPN no guarda sus proyecciones pasadas para todos los jugadores, así que este registro es la única forma de compararlas al final de la temporada.
5. La sección final evalúa lo registrado contra los resultados reales.

> Supuesto: el modelo predice los puntos *si el jugador juega*. El estado de lesión de ESPN se guarda para poder excluir a los que no jugaron al evaluar.

## 1. Setup

In [ ]:
from datetime import datetime, timezone

import polars as pl

from fantasy_ml import data, espn, scoring, model as M, evaluation as E, features as F, predictions_log as plog
from fantasy_ml.data import KEYS

SEASON = 2026
REFRESH = True  # siempre datos frescos: esta es la ejecución semanal

MODEL_CFG = data.load_config("model")
PARAMS = MODEL_CFG["params"]
CAL = {k: tuple(v) for k, v in data.load_config("ranges")["calibration"].items()}
SCORING = data.load_config("scoring")
pl.Config.set_tbl_rows(40)
print(f"Hiperparámetros congelados (ajustados con {MODEL_CFG['tuned_on']}) · versión del código: {plog.code_version()}")

In [ ]:
league = espn.connect(SEASON)
WEEK = league.current_week  # la semana que viene / en curso según ESPN

src = data.load_sources(refresh=REFRESH)
rosters = data.load_rosters_weekly(SEASON, refresh=REFRESH)

# ¿Están ya en nflverse todos los partidos jugados antes de esta semana?
tg = F.team_games(src["schedules"])
played_games = tg.filter(pl.col("season") == SEASON, pl.col("week") < WEEK, pl.col("team_score").is_not_null())["game_id"].unique()
stats_games = src["player_stats"].filter(pl.col("season") == SEASON)["game_id"].unique()
missing = set(played_games) - set(stats_games)
print(f"Semana a predecir: {SEASON} · {WEEK}")
print(f"Partidos jugados de {SEASON}: {len(played_games)} · sin estadísticas en nflverse todavía: {len(missing)}")
if missing:
    print(f"  ⚠ Faltan {sorted(missing)}: nflverse aún no los publica; las features de esos equipos van un partido atrasadas")
print(f"Rosters de nflverse disponibles para la semana {WEEK}: {rosters.filter(pl.col('week') == WEEK).height > 0}")

## 2. Features de la semana

Se usan las mismas funciones del paquete que en el notebook 02 y el backtest. Las filas de la semana a predecir tienen estadísticas nulas: sus features salen solo de los partidos ya jugados y su objetivo `y` queda vacío.

In [ ]:
ctx = F.game_context(tg)
ids = F.gsis_to_espn(src["playerids"])

base = F.offense_base(src["player_stats"], src["snap_counts"], src["opportunity"], src["playerids"])
up_off = F.upcoming_offense(rosters, tg, base, SEASON, WEEK)
off = F.build_offense(pl.concat([base, up_off], how="diagonal_relaxed"), ctx)

points_k = scoring.k_points(src["player_stats"], SCORING).join(ids, on="player_id", how="left")
up_k = F.upcoming_k(rosters, tg, points_k, SEASON, WEEK).join(ids, on="player_id", how="left")
k = F.build_k(pl.concat([points_k, up_k], how="diagonal_relaxed"), ctx)

points_dst = scoring.dst_points(src["team_stats"], src["schedules"], SCORING).join(F.dst_espn_ids(), on="team", how="left")
up_dst = F.upcoming_dst(tg, points_dst, SEASON, WEEK).join(F.dst_espn_ids(), on="team", how="left")
dst = F.build_dst(pl.concat([points_dst, up_dst], how="diagonal_relaxed"),
                  F.team_offense(tg, src["team_stats"], upcoming=(SEASON, WEEK)), ctx)

FRAMES = {"offense": off, "k": k, "dst": dst}
is_target = (pl.col("season") == SEASON) & (pl.col("week") == WEEK)
for g, df in FRAMES.items():
    print(f"{g:8s} filas a predecir: {df.filter(is_target).height:>4} · entrenamiento: {df.filter(pl.col('y').is_not_null()).height:,}")

## 3. Entrenar y predecir

In [ ]:
preds = []
for g, df in FRAMES.items():
    train = df.filter(pl.col("y").is_not_null())
    target = df.filter(is_target)
    assert train.filter(is_target).is_empty(), "La semana a predecir no puede estar en el entrenamiento"
    mdl = M.fit(train, g, PARAMS[g])
    q10, q90 = M.predict_range(tuple(M.fit_quantile(train, g, PARAMS[g], a) for a in M.QUANTILES), target, g, CAL)
    t = M.with_position(target, g)
    preds.append(t.select(
        "season", "week", group=pl.lit(g),
        entity_id=pl.col("team") if g == "dst" else pl.col("player_id"),
        espn_id=pl.col("espn_id") if "espn_id" in t.columns else pl.lit(None, dtype=pl.Int64),
        name=pl.concat_str("team", pl.lit(" D/ST")) if g == "dst" else pl.col("player_display_name"),
        position="position", team="team",
        opponent=pl.col("opponent") if g == "dst" else pl.col("opponent_team"),
        pred_model=pl.Series(M.predict(mdl, target, g)),
        pred_baseline=M.baseline(target, train, g),
        pred_q10=pl.Series(q10), pred_q90=pl.Series(q90)))
preds = pl.concat(preds, how="diagonal_relaxed")

# espn_id de QB/RB/WR/TE
preds = (preds.join(ids.rename({"player_id": "entity_id", "espn_id": "_eid"}), on="entity_id", how="left")
              .with_columns(espn_id=pl.coalesce("espn_id", "_eid")).drop("_eid"))
print(f"{preds.height:,} predicciones · sin espn_id: {preds['espn_id'].null_count()}")
assert (preds["pred_q10"] <= preds["pred_q90"]).all(), "P10 no puede ser mayor que P90"
print(f"Rango P10–P90 medio: {(preds['pred_q90'] - preds['pred_q10']).mean():.1f} puntos · "
      f"predicción dentro del rango: {((preds['pred_model'] >= preds['pred_q10']) & (preds['pred_model'] <= preds['pred_q90'])).mean():.1%}")

## 4. Proyecciones de ESPN y hora de inicio

ESPN da la proyección de la semana de los jugadores con roster (box scores) y de los agentes libres más rostereados de cada posición. Los jugadores muy marginales pueden quedar sin proyección.

In [ ]:
espn_week = espn.week_projections(league, WEEK)
kick = tg.filter(pl.col("season") == SEASON, pl.col("week") == WEEK).select("team", kickoff_utc=plog.kickoff_utc(tg))

rows = (preds.join(espn_week.select("espn_id", "espn_projection", "espn_injury_status", "on_my_roster", "my_slot"),
                   on="espn_id", how="left")
             .join(kick, on="team", how="left")
             .with_columns(on_my_roster=pl.col("on_my_roster").fill_null(False)))
print(f"Con proyección de ESPN: {rows['espn_projection'].is_not_null().sum():,} de {rows.height:,}")
print(f"Partidos de la semana ya iniciados (no se registran): {(rows['kickoff_utc'] <= datetime.now(timezone.utc)).sum()}")

## 5. Registrar

In [ ]:
logged = plog.append(rows)
print(f"✓ {logged.height:,} predicciones agregadas a data/predictions_log/{SEASON}.csv "
      f"(versión {logged['code_version'][0] if logged.height else '-'})")
print(f"  excluidas por partido ya iniciado: {rows.height - logged.height}")
print("  Recuerda hacer commit del registro para fechar las predicciones.")

### Mi roster esta semana

In [ ]:
(rows.filter("on_my_roster")
     .select("my_slot", "name", "position", "team", "opponent", "espn_injury_status",
             pl.col("pred_model").round(1), pl.col("pred_q10").round(1), pl.col("pred_q90").round(1),
             pl.col("pred_baseline").round(1), "espn_projection")
     .sort(pl.col("my_slot").is_in(["BE", "IR"]), pl.col("pred_model"), descending=[False, True]))

## 6. Evaluación del registro

Cruza la **última predicción previa al inicio** de cada partido con los puntos reales: nflverse para QB/RB/WR/TE, y el cálculo validado de K y D/ST. Los jugadores que no jugaron no tienen resultado y quedan fuera, porque el modelo supone que el jugador juega. La comparación con ESPN usa solo los jugadores con proyección de ESPN, con las mismas filas para los tres métodos.

In [ ]:
log = plog.latest_pregame(plog.read(SEASON))
actual = pl.concat([
    base.filter(pl.col("season") == SEASON).select(*KEYS, group=pl.lit("offense"), entity_id="player_id", y="fantasy_points_ppr"),
    points_k.filter(pl.col("season") == SEASON).select(*KEYS, group=pl.lit("k"), entity_id="player_id", y="fantasy_points"),
    points_dst.filter(pl.col("season") == SEASON).select(*KEYS, group=pl.lit("dst"), entity_id="team", y="fantasy_points"),
])
ev = log.join(actual, on=["season", "week", "group", "entity_id"], how="inner")
print(f"Predicciones registradas (última pre-partido): {log.height:,} · con resultado: {ev.height:,} · "
      f"semanas evaluables: {sorted(ev['week'].unique().to_list())}")
if ev.is_empty():
    print("Aún no hay semanas registradas con resultado: vuelve a ejecutar cuando se jueguen.")

In [ ]:
if not ev.is_empty():
    with_espn = ev.filter(pl.col("espn_projection").is_not_null())
    display(E.metrics(with_espn.rename({"pred_model": "modelo", "pred_baseline": "baseline", "espn_projection": "espn"}),
                      ["position"], models=("modelo", "baseline", "espn")))
    display(E.bootstrap_mae_diff(with_espn, a="pred_model", b="espn_projection"))

### Cobertura de los rangos P10–P90 en 2026

Solo filas registradas con rango (desde el 23-09-2026). Objetivo: 80% dentro, 10% debajo y 10% encima.

In [ ]:
rng = ev.filter(pl.col("pred_q10").is_not_null())
if rng.is_empty():
    print("Aún no hay partidos jugados con rango registrado.")
else:
    display(M.range_coverage(rng.rename({"pred_q10": "q10", "pred_q90": "q90"}), ["position"]))